____
### 1. Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal


____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes:
1. observation space, 5 numbers as an array [Leftover Stock, Days Left, Last known demand]
2. action space (price of good, number over a continuouse range from 5 to 50)
3. max steps (days of simulation)
4. cost (cost incurred to obtain 1 unit)


In [ ]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        reward = reward / 100.0
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        if truncated and self.inventory > 0:
            reward -= self.inventory * 2.0  # $2 penalty per unsold unit
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory / self.max_inventory, # 0 to 1
                        (self.max_steps - self.step_count) / self.max_steps, # 0 to 1
                        self.last_demand / self.max_inventory] # 0 to 1
                        , dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 0.8
        noise = np.random.normal(0, 2) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment

In [3]:
env = DynamicPricingEnv()

In [4]:
obs, info = env.reset()
print(f"Observation space representing: [stock left, days left, last known sold]: {obs}")
print(f"Extra info dictrionary: {info}")

obs, reward, terminated, truncated, info = env.step([10.0])
env.get_latest()
print(reward)
print(f"{terminated}, {truncated}")

Observation space representing: [stock left, days left, last known sold]: [1. 1. 0.]
Extra info dictrionary: {}
Leftover Stock: 0.6700000166893005 units, Days Left 0.9666666388511658, Sold Units: 0.33000001311302185
1.65
False, False


____
### 3. Deciding which model to use 
- A standard Q-table cannot be used as the action space (price of good) is a continuous number instead of a discrete number

#### 3.1 Proximal Policy Optimization (PPO) Model
- a policy gradient method which directly learns "Given this state, what price should I output"
- the "Proximal" part of the PPO model adjusts the actions in small amounts to find the optimal policy 
- therefore, PPO Models limits how drastically the policy changes each update to prevent unstable training

##### PPO Architecture
- PPO uses 2 networks
1. Actor Network
    - Outputs the pricing policy
    - Given a state, feeds into a neutal network and outputs a pricing distribution
    - `state → neural network → price distribution`
    - The agent then samples prices around that range

2. Critic Network
    - Estimates the future rewards
    - Asks "How profitable is this situation" and helps the Actor Network improve

##### PPO Flow
`Observe market → Choose price → Simulate customer response → Get profit reward → Update pricing policy slightly`


#### 3.2 Twin Delayed Deep Deterministic Policy Gradient (TD3)
- Designed specefically for continuous action space, precise control and stable deep Q-learning
- instead learning "what action should I take?" the model learns "how good is a particular action"

##### TD3 Architecture
- TD3 uses 3 networks
1. Actor Network
    - Outputs a distribution over actions
    - `state → distribution`
    - Samples from the distribution to create exploration

2. Critic Network (2 critic networks)
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as PPO)

3. Replay Buffer
    - To store experiences and reuse them, making the TD3 highly sample efficient
    - Allow TD3 to learn from past experiences repeatedly

4. Target Networks

##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`
- Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)



#### 3.3 Soft Actor-Critic (SAC)
- Considered one of the strongest RL algorithms for continuous control
- Combines actor-critic learning, entropy maximisation and off-policy training
- Tries to maximise both `Reward` and `Exploration` instead of only `profit`
- Entropy refers to the epsilon (randomness of the actions) therefore it encourages the agent to keep exploring pricing options, preventing the model from becoming too deterministic 


##### SAC Architecture
- SAC uses 3 networks
1. Actor Network
    - Outputs the exact price
    - `state → price`

2. Critic Network (2 critic networks)
    - The "Twin" part is in refernce to the 2 critic networks
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` 

##### SAC Flow
- `Observe State → Sample action from policy distribution → receive reward → update critic → update actor → encourage exploration through entropy bonus`
- Reward is evaluated as `total reward = reward + entropy bonus` and to encourage exploration

____
### 4. Training the PPO Model
- In the spirit of learning, I will be training a PPO model first before training a TD3 model and a SAC model

#### 4.1 Establishing Architecture
- Actor and Critic networks are first created
    - Actor and Critic networks are used to observe the state and output the action and critic values, no learning logic implemented yet
- Create RolloutBuffer to store data and translate the data into learning signals 
    - Data is converted to learning signals, networks do not learn yet
- `ppo_update()` function converts teh learning signals and updates the networks via gradient descent
    - computes policy loss, value loss and entropy loss
- `train()` function combines the network, rollout buffer and `ppo_update()` to train the model 

#### 4.2 Creating Actor and Critic Networks

In [5]:
class ActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        # self.backbone acts as a shared extractor used by both the actor and critic network
        self.backbone = nn.Sequential( #nn.Sequential runs the layers in order, linear -> Tanh -> linear -> tanh
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(64, action_dim) 
        self.log_std = nn.Parameter(torch.ones(action_dim) * 2.0) 
        self.critic = nn.Linear(64, 1)

    def forward(self, obs): #needs to be overridden
        features = self.backbone(obs) #vector of 128 values
        mean = self.actor_mean(features) #obtain the action 
        std = self.log_std.exp().expand_as(mean) #obtain the std dev and fits the shape with with mean 
        critic_val = self.critic(features) #obtain the critic value
        return mean, std, critic_val
    
    def get_action(self, obs): #creating action based off the network
        mean, std, value = self.forward(obs) #calling forward to obtain values from actor and critic network
        dist = Normal(mean, std) #creates normal distribution
        action = dist.sample() #samples the distribution
        log_prob = dist.log_prob(action).sum(dim=-1) #obtains sum of log distribution (exp below)
        return action, log_prob, value.squeeze(-1) #converts the value into a scalar

    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1) #calculates the entropy of the normal distribution, how random or uncertain the distribution is 
        # entropy is summed up over the last dimension
        return log_prob, value.squeeze(-1), entropy

##### 4.21 Explanation of Code:
```python
self.backbone = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh()
        )
```
- `nn.sequential()` ensures that the following layers run in sequence
- `nn.Linear(obs_dim, 64)` -> y = Wx + b
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.Tanh()` -> y = tanh(x)
    - x is a `64 x 1 vector` that is the result from the linear vector
    - y is the resultant `64 x 1 vector` from applying the tanh() function on every single value in the original vector
    - this function introduces non-linearlity and squashes every value to be within the range (-1, 1)
    - the introduction of non-linearity allows the model to learn non-linear behaviours
- `nn.Linear(64, 64)` -> y = Wx + b
    - uses the non-linear outputs from the previous layers but turns them into more sophisticated outputs using weights and biases
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer

```python
self.actor_mean = nn.Linear(64, action_dim) 
self.log_std = nn.Parameter(torch.ones(action_dim) * 2.0)
self.critic = nn.Linear(64, 1)
```
- `self.actor_mean = nn.Linear(64, action_dim)`
    - Makes use of the 64 outputs from the backbone to derive the outputs in the action dimension (mean of the price)
- `self.log_std = nn.Parameter(torch.zeroes(action_dim))`
    - std deviation measures the randomness of the distributionm, log(std) is used so that the value is not -ve, later converted using `exp()`
    - wrapping it in `nn.Parameter()` means that the value should be learning during training
    - this makes it such that the PPO model learns during training what is the appropriate level of exploration
- `self.critic = nn.Linear()`
    - Makes use of the 64 outputs from the backbone to derive 1 value, which represents the future reward expected from the current state

- `log_prob = dist.log_prob(action).sum(dim=-1)`
    - `dist.log_prob(action)` obtains the log_probability of the action within the distribution
    - `.sum(dim=-1)` sums it along the last dimension
    - instad of multiplying individual probabilities, it adds up `log(prob)` instead 

##### 4.22 Over-Arching View:
- observation (vector of 3 values) -> self.backbone(vector of 64 values) 
- The observation in the form of 64 values is then fed into the actor network and critic network
- ie `self.backbone(vector of 64 values) -> action (1 value)` and `self.backbone(vector of 64 values) -> future reward expected (1 value)`

##### 4.3 Creating RolloutBuffer
- Acts as a container to store past experiences and convert them into learning signals
- `RolloutBuffer` converts raw experiences into 3 learning signals:
    1. Advantages: How much better an action was than expected
    2. Returns: The total expected future reward
    3. Normalized Advantages: Standardized advantages for stable training

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.clear() #delegates initialisation to the clear() method

    def clear(self): #resets all lists to empty
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []

    def add(self, obs, action, log_prob, reward, value, done): #appends one round of experiences
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95): #compute generalized advantage est
        advantages = [] 
        gae = 0.0
        values = self.values + [last_value] # adds one extra value to compute next-step difference
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns

##### 4.31 Explanation of code

```python
def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95):
    advantages = [] 
    gae = 0.0
    values = self.values + [last_value] 
    for t in reversed(range(len(self.rewards))): 
        delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
        gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
        advantages.insert(0, gae)
    returns = [adv + val for adv, val in zip(advantages, self.values)]
    return advantages, returns
```
- `compute_returns()` is computes the Advantages ("How much better an action was than expected") and the returns ("total expected future rewards")
- Temporal Diff (TD) error: `delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]` measures the diff in outcome compared to the expected outcome (measuring suprise)
- Equation generalises to `TD = (actual outcome) - (expected outcome)`
    - delta < 0, worse than expected, delta > 0, better than expected
    - `self.rewards[t]` is the immediate reward from action 
    - `gamma` is the discount factor, measures how much the model cares about future rewards
    - `values[t + 1]` is the predicted future value, the critic's estimate of future reward after 1 step 
    - `gamma * values[t + 1]` is the expected future rewards, multiplying by `gamma` discounts the expected future rewards 
    (expected future rewards are slightly less vaulable)
    - `values[t]` is baseline prediction before seeing the outcome
    - `1 - self.dones[t]` acts as a switch, if the episode has not terminated at the step t, `self.dones[t] = 0`, if it has terminated, `self.dones[t] = 1` and `1 - self.dones[t] = 0`
    - When episode has ended, `1 - self.dones[t] = 0` and future term of `gamma * values[t + 1] * (1 - self.dones[t]) = 0`
    - Therefore, `delta = self.rewards[t] - values[t]` since there is no more `expected future rewards` and the difference in expectation is simply the rewards up to that point, `self.rewards[t]`, minus expected rewards `values[t]`

- Generalized Advantage Estimation: `gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae` which is a smoothened estimate of "how good was this action"
- Equation generalises to `gae = current TD error + discounted future gae`
    - starts out with the current step's TD error
    - adds the discounted future TD errors = `gamma * gae_lambda * (1 - self.dones[t]) * gae`
    - when the episode ends, `self.dones[t] = 1` and `1 - self.dones[t] = 0` and the future TD errors = 0 (no more future for the episode since it terminated) 

- `advantages.insert(0, gae)`
    - inserts the calculated gae at the **front** of the `advantages` array
    - when the loop runs in reverse during `for t in reversed(range(len(self.rewards))):` the first iteration takes the last timestep. with the calculated gae inserted at the front, the advantages are in forward order

- `returns = [adv + val for adv, val in zip(advantages, self.values)]` : equivalent to doing:
    ```python
    returns = []
    for adv, val in zip(advantages, self.values):
        r = adv + val
        returns.append(r)
    ```
    - for each step, returns = advantage + value estimate
    - adding it to the array saved as returns 

``` python
def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns
```
- `torch.tensor(data).to(device)`
    - Creates a new tensor from python data (list, arrays, numbers), `data -> tensor`
    - used for `obs`, `advantages` and `returns` as they are all raw lists of data
-  `torch.stack(tensors, dim = 0)`
    - joins multiple existing tensors along a new dimension
    - used for `actions` and `log_probs` as `self.actions` and `self.log_probs` is a list of tensors already
- `to(device)` is used to move a tensor to a specified device (CPU or GPU)
- `advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)`
    - `advantages - advantages.mean()` subtracts the mean from every value such that it is values in the tensors are now **centred around zero** 
    - `/(advantages.std() + 1e-8)` divides every value from the std deviation 
    - this process normalises all data within the tensor (tensor supports element-wise operations)


##### 4.32 Over-Arching View:
1. `add()` method
   - raw data is collected: observations, actions, log probabilities, rewards, values, and done flags
   - All stored as lists in the buffer

2. `compute_returns()` method
   - Compute TD errors (`delta`) by comparing actual outcomes to critic predictions
   - Accumulate GAE backward through time to get smooth advantage estimates
   - Combine advantages with value estimates to get target returns for the critic

3. `to_tensors()` method
   - Convert all lists to PyTorch tensors
   - Normalize advantages to have mean=0 and std=1 for stable training
   - Move tensors to the correct device (CPU or GPU)

4. **Learning Signal Output**
   - `advantages`: How much better/worse each action was compared to expected (used by actor)
   - `returns`: Target value estimates (used by critic)
   - Both aligned with original observations and actions for supervised learning

##### 4.4 Instantiate functions to update PPO

In [7]:
def ppo_update(model, optimizer, obs, actions, old_log_probs,advantages, returns, 
               clip_range=0.2, ent_coef=0.05, vf_coef=0.5, n_epochs=10, batch_size=64):
    total_steps = obs.shape[0] 
    for _ in range(n_epochs):
        indices = torch.randperm(total_steps)
        for start in range(0, total_steps, batch_size):
            idx = indices[start : start + batch_size]
            new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])
            ratio = (new_log_probs - old_log_probs[idx]).exp()
            adv = advantages[idx]
            policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
            value_loss = nn.functional.mse_loss(values, returns[idx])
            entropy_loss = -entropy.mean()
            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

##### 4.41 Explanation of code
- Required Parameters:
    1. `model` : ActorCritic neural network being trained
    2. `optimizer` : The optimizer that applies weight updates via backpropagation (adjusting weights of each matrix using backprop)
    3. `obs` : observations collected during rollout, `512x3 matrix`, converted to tensor (3 cols as observation space is 3 dimension)
    4. `actions` : actions taken during rollout, `512x1 vector`, converted to tensor (1 col as action space is just 1 value)
    5. `old_log_probs` : log prob of actions under the old policy (before update), used to compute importance sampling ratio
    6. `advantages` : normalized advantage estimates (array with 512 entries), tells us how good each action was compared to expected 
    7. `returns` : target returns for the critic (array with 512 entries), what the critic model should predict
- 512 -> max of 512 timesteps in an episode
- Hyperparameters:
    1. `clip_range=0.2` : clipping parameter, restricting the policy ration to [1-0.2, 1+0.2] = [0.8, 1.2], to prevent drastic policy changes
    2. `ent_coef=0.05` : weight on exploration bonus, provides incentive to explore 
    3. `vf_coef=0.5` : value function coefficient, the weight on critic loss - How important is predicting returns accurately
    4. `n_epochs=10` : number of passes through the data, extracting more learning signals each epock
    5. `batch_size=64` : batch size for gradient updates, process 64 timesteps at a time
    
- `total_steps = obs.shape[0]` 
    - `.shape` gives (rows, cols), therefore `obs.shape[0]` = rows of data = number of timesteps

```python
for _ in range(n_epochs):
    indices = torch.randperm(total_steps)
    for start in range(0, total_steps, batch_size):
```
- iterate through set number of times according to hyperparameters set
`indices = torch.randperm(total_steps)`
    - creates a tensor of the numbers from 0 to total_steps-1 in a **random order**
    - this shuffles the rollout data before training for each iteration so that the PPO does not see the data in the same order each time
    - this breaks any ordering bias and makes mini-batches random
`for start in range(0, total_steps, batch_size)`
    - iterates from 0 to total_steps, jumping by `batch_size` at once


- `idx = indices[start : start + batch_size]` : slices the array in chunks of length equal to `batch_size`, becomes the current batch fof sample indices
- `new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])` : evaluates the model based off the small batch selected
- `ratio = (new_log_probs - old_log_probs[idx]).exp()` : computes the PPO probability ratio, measuring how much the new policy changed compared to the old policy for those same actions (element wise operation for the whole array)
    - ratio = 1, policy is unchanged
    - ratio > 1, the new policy assigns higher probability to the action ()
    - ratio < 1, the new policy assigns lower probability to the action
- `adv = advantages[idx]` : obtains the advantage values for this batch of values
- ``` python 
    policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()
    ```
- `ratio * adv` : element-wise operation between importance-sampling ratio and the advantage array for this batch, `adv`
- `torch.clamp(ratio, 1 - clip_range, 1 + clip_range)` : `torch.clamp(tensor, min, max)` clips all the values inside the ratio tensor to within the range of (1 - clip_range, 1 + clip_range)
    - if value < min or value > max , the value is set to min / max
    - `* adv` to perform element-wise operation between importance-sampling ratio and the advantage array for this batch
- `torch.min(tensor1, tensor2)` : compares the 2 tensors element-wise and extracts the lower value at each position, constructing a new tensor out of it
- `.mean()` : averages all values of the tensor to obtain the mean 
- calculating policy loss:
    - finding the **mean advantage value** after multiplying by ratio element-wise and then **limiting it** to within the range `(1-clip_range, 1+clip_range)`
    - multiplying by `-` then obtains the `policy loss` 

- `value_loss = nn.functional.mse_loss(values, returns[idx])` : the Mean Square Error between critic predictions and targets
    - `nn.functional.mse_loss(input_tensor, target_tensor)` finds the mean squared error between every value in the input and the target tensor
    - use `reduce = none` to get the output as a tensor, else it returns the average of all the MSE values
- `entropy_loss = -entropy.mean()` : multiplies the mean entropy by -1, turning the maximise entropy into a minimization objective
    - when entropy increases, loss decreases which is what gradient descent wants
- `loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss`
    - formula generalises to : overall loss = policy loss + overall value loss + overall entropy loss 
    - `overall value loss = vf_coef * value_loss`, where `vf_coef` scales the critic term in thte total loss equation (how strongly gradient descent prioritizes reducing the value, critic's accuracy, relative to policy and entropy)
    - `overall entropy loss = ent_coef * entropy_loss`, where `ent_coef` controls how strongly exploration is encouraged (how strongly gradient descent prioritizes exploration, relative to policy and value)
- `optimizer.zero_grad()` : clears the gradients stored inside the optimizer before calculations are done 
- `loss.backward()` : with loss represented as a function of policy loss, value loss and entropy loss, ie L = f(p,v,e)
    - calling `.backward()` differentiates loss wrt. p, v and e and stores it inside each parameter, which can be accessed with `.grad()`
- `nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)` : rescales `.grad()` in place if the total norm > max_norm, so that the parameter update magnitude is limited when `optimizer.step()` runs
- `optimizer.step()` : computes updates for each parameter's rule within the optimizer and updates it
    - this thus makes the new weight value of the neural network the model's current parameters    
    - this updates the whole network, meaning the backbone network, actor and critic network.

##### 4.42 Over-Arching View:
- The env produces 3‑D observations that a shared backbone encodes into features consumed by an `ActorCritic` network (actor outputs a Gaussian price policy; critic predicts state value).
- Episodes are collected into a `RolloutBuffer` (obs, actions, log-probs, rewards, values, dones); GAE computes advantages and returns from those rollouts.
- `ppo_update()` runs multiple epochs of minibatch updates with the clipped policy objective, value MSE, and entropy bonus to update the network.
- Repeat collect → compute (advantages/returns) → update until total timesteps, producing a trained pricing policy.

##### 4.5 Instantiate function to train model

In [8]:
def train(total_timesteps=200_000, n_steps=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    env = DynamicPricingEnv()
    obs_dim = env.observation_space.shape[0] # 3
    action_dim = env.action_space.shape[0] # 1
    model = ActorCritic(obs_dim, action_dim).to(device) # moves all of the model's tensors to the device
    optimizer = optim.Adam(model.parameters(), #register all params, backkbone, actor head, log_std, critic
                            lr=3e-4) # learning rate set at 3e-4
    buffer = RolloutBuffer()
    obs, _ = env.reset()
    episode_reward = 0
    episode_count = 0
    timestep = 0
    while timestep < total_timesteps: # runs until timestep budget is exhausted
        buffer.clear() #reset the buffer at the start of every rollout: old experience thrown away as they were collected under a previous version of the policy
        for _ in range(n_steps):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            with torch.no_grad(): #actions within this block temporarilty disables autograd and do not track history or build computation graphs
                action, log_prob, value = model.get_action(obs_tensor)
            action_np = action.cpu().numpy()[0]
            action_np = np.clip(action_np, 5.0, 50.0) 
            next_obs, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated
            buffer.add(obs = obs,
                       action = action.squeeze(0).cpu(),
                       log_prob = log_prob.squeeze(0).cpu(),
                       reward = reward,
                       value = value.squeeze(0).cpu().item(),
                       done = float(done))
            episode_reward += reward
            obs = next_obs
            timestep += 1
            if done:
                episode_count += 1
                if episode_count % 20 == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0
                obs, _  = env.reset()
        with torch.no_grad():
            last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            _, _, last_value = model.get_action(last_obs)
            last_value = last_value.squeeze(0).cpu().item()
        advantages, returns = buffer.compute_returns(last_value)
        obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
        ppo_update(model, optimizer, obs_t, act_t, lp_t, adv_t, ret_t)
    # torch.save(model.state_dict(), "ppo_pricing.pth")
    print("Training complete.")
    return model


##### 4.51 Explanation of code

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")
env = DynamicPricingEnv()
obs_dim = env.observation_space.shape[0] # 3
action_dim = env.action_space.shape[0] # 1
model = ActorCritic(obs_dim, action_dim).to(device) 
optimizer = optim.Adam(model.parameters(), lr=3e-4) 
```
- `model = ActorCritic(obs_dim, action_dim).to(device)`
    - instantiate a model using the `obs_dim` and `action_dim` of the environment
    - using `.to(device)` moves the tensor of the model to `device`

- `optimizer = optim.Adam(model.parameters(), lr = 3e-4)`
    - creates a **Adaptive Moment Estimation** optimizer, updating the network's weights
    - runs 2 running statistics per parameter:
        1. The mean of gradients
        2. The mean of squared gradients 
    - `model.parameters()` as an input : registers all parameters of the model in the optimizer
    - `lr = 3e-4` : learning rate for the PPO, cannot be too low or too high

``` python
while timestep < total_timesteps:
    buffer.clear()
```
- outer loop runs until the total timestep budget is exhausted, `buffer.clear()` resets the buffer at the start of every rollout
- This ensures that old experiences are thrown away as they are collected under a previous version of the policy and no longer valid for the current update

```python
for _ in range(n_steps):
    obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
```
- `torch.tensor(obs, dtype=torch.float32)` : converts the NumPy array (obs) into a PyTorch Tensor
    - datatype set to `float.32` as that is the type used by neural networks
- `unsqueeze(0)` : inserts a dimension at postion 0
    - `nn.Linear` layers expect inputs shaped as `[batch_size, features]` while the `obs` tensor is shaped `[3]`
    - therefore, `.unsqueeze(0)` inserts a dimension at position 0, making it `[1, 3]`, a batch of size 1
    - this is required for matrix multiplication


```python
with torch.no_grad():
    action, log_prob, value = model.get_action(obs_tensor)
```
- `with torch.no_grad()`
    - disables gradient computation for the methods inside, which is `action, log_prob, value = model.get_action(obs_tensor)`
    - in context here, it disables computation of gradient for `model.get_action(obs_tensor)` because this portion is just meant for data collection
    - therefore, there is no need to update the model as it is simply used to make decisions 
    - this is more efficient for computation

```python
action_np = action.cpu().numpy()[0]
action_np = np.clip(action_np, 5.0, 50.0) 
next_obs, reward, terminated, truncated, _ = env.step(action_np)
done = terminated or truncated
```
- `action.cpu().numpy()[0]` : `.cpu().numpy()` moves the tensor to the cpu so that numy can access it and converts it to a numpy array
    - `[0]` is used to remove the batch dimension, converting it from an array shaped `[1, 1]` (batch of 1, dimension 1) to an array of shape `[1]`, a 1-element array containing the price
- `np.clip(5.0, 50.0)` : limits the action to be a price within 5 and 50 since the range is theoratically unlimited
    - clip the range after sampling rather than sampling from a bounded distribution
- `next_obs, reward, terminated, truncated, _ = env.step(action_np)` : obtains the state after the nn provides an action

```python
buffer.add(obs = obs,
            action = action.squeeze(0).cpu(),
            log_prob = log_prob.squeeze(0).cpu(),
            reward = reward,
            value = value.squeeze(0).cpu().item(),
            done = float(done))
```
- updating the buffer with all new states, applying `.squeeze(0)` to `action`, `log_prob` and `value` to adjust its shape and converts `done` to a float for computation purposes within buffer

```python
with torch.no_grad():
    last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
    _, _, last_value = model.get_action(last_obs)
    last_value = last_value.squeeze(0).cpu().item()
advantages, returns = buffer.compute_returns(last_value)
obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)
```
- This step is to process any partial episodes at the end of the 512 timestep limit.
- e.g. if each episode take 100 steps, the last 12 steps will be an incomplete episode. Therefore, this step will process the last known observation 
    - This processing is done using `with torch.no_grad()` to turn off the computation of gradients so that the model is not updated using incomplete episode data
    - next two lines are then used to obtain the price suggested by the model 
- `advantages, returns = buffer.compute_returns(last_value)` : use the last observation, the incomplete data to calculate the advantages and returns
- `obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(advantages, returns, device)` : converts the data stored in the buffer to learning signals 





In [9]:
model = train(total_timesteps=200_000)

Training on: cpu
Timestep      62 | Episode   20 | Reward:     1.69
Timestep     127 | Episode   40 | Reward:     0.00
Timestep     188 | Episode   60 | Reward:     1.59
Timestep     253 | Episode   80 | Reward:     1.43
Timestep     313 | Episode  100 | Reward:     0.76
Timestep     376 | Episode  120 | Reward:     0.00
Timestep     436 | Episode  140 | Reward:     0.00
Timestep     498 | Episode  160 | Reward:     1.11
Timestep     563 | Episode  180 | Reward:     0.95
Timestep     626 | Episode  200 | Reward:     0.00
Timestep     691 | Episode  220 | Reward:     2.55
Timestep     760 | Episode  240 | Reward:     4.44
Timestep     824 | Episode  260 | Reward:     0.73
Timestep     890 | Episode  280 | Reward:     4.15
Timestep     952 | Episode  300 | Reward:     1.58
Timestep    1018 | Episode  320 | Reward:     0.00
Timestep    1085 | Episode  340 | Reward:     0.00
Timestep    1153 | Episode  360 | Reward:     0.95
Timestep    1222 | Episode  380 | Reward:     0.00
Timestep    12

##### 4.6 Instantiate functions to evaluate model

In [10]:
def evaluate_model(model, n_episodes=50, deterministic=True):
    device = next(model.parameters()).device
    model.eval()
    episode_rewards = []
    ending_inventory = []
    steps_taken = []
    for _ in range(n_episodes):
        env = DynamicPricingEnv()
        obs, _ = env.reset()
        done = False
        total_reward = 0.0
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, #convert state to a batched tensor 
                                      device=device).unsqueeze(0) #unsqueeze.(0) inserts a new dimension of size 1 at the 0th position
            with torch.no_grad(): #temporarily disables autograd so that operations inside do not track history or build computation graphs
                mean, std, _ = model.forward(obs_tensor)
                if deterministic:
                    action = mean
                else:
                    action = Normal(mean, std).sample()
            action_np = action.squeeze(0).detach().cpu().numpy()
            action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
            obs, reward, terminated, truncated, _ = env.step(action_np)
            total_reward += reward
            done = terminated or truncated
        episode_rewards.append(total_reward)
        ending_inventory.append(env.inventory)
        steps_taken.append(env.step_count)
    results = {
        "episode_rewards": episode_rewards,
        "mean_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "min_reward": float(np.min(episode_rewards)),
        "max_reward": float(np.max(episode_rewards)),
        "mean_ending_inventory": float(np.mean(ending_inventory)),
        "mean_steps": float(np.mean(steps_taken))}
    print(f"Episodes: {n_episodes}")
    print(f"Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
    print(f"Reward range: [{results['min_reward']:.2f}, {results['max_reward']:.2f}]")
    print(f"Mean ending inventory: {results['mean_ending_inventory']:.2f}")
    print(f"Mean steps per episode: {results['mean_steps']:.2f}")

    return results

In [11]:
eval_results = evaluate_model(model, n_episodes=100, deterministic=True)

Episodes: 100
Mean reward: 38.94 +/- 0.14
Reward range: [38.49, 39.20]
Mean ending inventory: 0.00
Mean steps per episode: 25.41


##### 4.7 Storing the model as a class

In [19]:
class PPOAgent():
    def __init__(self, name: str = "bot", NN=None, env : gym.Env = None): #Store a PyTorch model and device for inference / persistence
        self.name = name
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        if env is not None:
            self.env = env
        else: 
            self.env = DynamicPricingEnv()
        if NN is None:
            self.NN = ActorCritic(self.env.observation_space.shape[0], self.env.action_space.shape[0]).to(self.device)
        else:
            self.NN = NN.to(self.device)
        self.buffer = RolloutBuffer()
        self.optimizer = optim.Adam(self.NN.parameters(), lr=3e-4)

    def train(self, total_timesteps=200_000, n_steps=512):
        self.NN.train() #set to training mode
        obs, _ = self.env.reset()
        episode_reward, episode_count = 0 , 0
        timestep = 0
        while timestep < total_timesteps: # runs until timestep budget is exhausted
            self.buffer.clear() #reset the buffer at the start of every rollout
            for _ in range(n_steps):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    action, log_prob, value = self.NN.get_action(obs_tensor)
                action_np = action.cpu().numpy()[0]
                action_np = np.clip(action_np, 5.0, 50.0)
                next_obs, reward, terminated, truncated, _ = self.env.step(action_np)
                done = terminated or truncated
                self.buffer.add(obs = obs,
                        action = action.squeeze(0).cpu(),
                        log_prob = log_prob.squeeze(0).cpu(),
                        reward = reward,
                        value = value.squeeze(0).cpu().item(),
                        done = float(done))
                episode_reward += reward
                obs = next_obs
                timestep += 1
                if done:
                    episode_count += 1
                    if episode_count % 20 == 0:
                        print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                            f"Reward: {episode_reward:>8.2f}")
                    episode_reward = 0
                    obs, _  = self.env.reset()
            with torch.no_grad():
                last_obs = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
                _, _, last_value = self.NN.get_action(last_obs)
                last_value = last_value.squeeze(0).cpu().item()
            advantages, returns = self.buffer.compute_returns(last_value)
            obs_t, act_t, lp_t, adv_t, ret_t = self.buffer.to_tensors(advantages, returns, self.device)
            self.ppo_update(obs_t, act_t, lp_t, adv_t, ret_t)
        print("Training complete.")
        return self

    def act(self, obs, deterministic=True):
        obs_tensor = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)
        self.NN.eval()
        with torch.no_grad():
            mean, std, _ = self.NN.forward(obs_tensor)
            if deterministic:
                action = mean
            else:
                action = Normal(mean, std).sample()
        action_np = action.squeeze(0).cpu().numpy()
        action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
        return action_np

    def play(self, deterministic=True, render=False):
        obs, _ = self.env.reset()
        done = False
        total_reward = 0.0
        total_revenue = 0.0
        steps = 0
        trajectory = []
        while not done:
            action = self.act(obs, deterministic=deterministic)
            # ensure a plain float price and log demand before stepping (demand is stochastic)
            price = float(action[0]) if hasattr(action, '__iter__') else float(action)
            demand = self.env.demand(price)
            sold = min(demand, int(self.env.inventory))
            revenue = price * sold
            total_revenue += revenue
            next_obs, reward, terminated, truncated, _ = self.env.step([price])
            done = terminated or truncated
            trajectory.append((obs, float(price), int(demand), float(reward)))
            # print step-level info
            print(f"Demand: {demand}, Price: {price:.2f}, Reward: {reward:.2f}, Sold: {sold}, Revenue: {revenue:.2f}")
            total_reward += reward
            steps += 1
            obs = next_obs
            if render:
                self.env.get_latest()
        print(f"Episode finished — Reward: {total_reward:.2f}, Steps: {steps}, Ending inventory: {int(self.env.inventory)}")
        print(f"Total revenue generated: ${total_revenue:.2f}")


    def evaluate_model(self, n_episodes=50, deterministic=True):
        device = next(self.NN.parameters()).device
        self.NN.eval()
        episode_rewards, ending_inventory = [], []
        steps_taken = []
        for _ in range(n_episodes):
            env = DynamicPricingEnv()
            obs, _ = env.reset()
            done = False
            total_reward = 0.0
            while not done:
                obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    mean, std, _ = self.NN.forward(obs_tensor)
                    if deterministic:
                        action = mean
                    else:
                        action = Normal(mean, std).sample()
                action_np = action.squeeze(0).detach().cpu().numpy()
                action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)
                obs, reward, terminated, truncated, _ = env.step(action_np)
                total_reward += reward
                done = terminated or truncated
            episode_rewards.append(total_reward)
            ending_inventory.append(env.inventory)
            steps_taken.append(env.step_count)
        results = {"episode_rewards": episode_rewards, "mean_reward": float(np.mean(episode_rewards)),
            "std_reward": float(np.std(episode_rewards)),"min_reward": float(np.min(episode_rewards)),
            "max_reward": float(np.max(episode_rewards)),"mean_ending_inventory": float(np.mean(ending_inventory)),
            "mean_steps": float(np.mean(steps_taken))}
        return results

    #helpers:
    def ppo_update(self, obs, actions, old_log_probs, advantages, returns, 
               clip_range=0.2, ent_coef=0.05, vf_coef=0.5, n_epochs=10, batch_size=64):
        total_steps = obs.shape[0]
        self.NN.train()
        for _ in range(n_epochs):
            indices = torch.randperm(total_steps)
            for start in range(0, total_steps, batch_size):
                idx = indices[start : start + batch_size]
                new_log_probs, values, entropy = self.NN.evaluate(obs[idx], actions[idx])
                ratio = (new_log_probs - old_log_probs[idx]).exp()
                adv = advantages[idx]
                policy_loss = -torch.min(ratio * adv, torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv).mean()
                value_loss = nn.functional.mse_loss(values, returns[idx])
                entropy_loss = -entropy.mean()
                loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.NN.parameters(), max_norm=0.5)
                self.optimizer.step()


In [20]:
# Example usage (commented to avoid long runs)
agent = PPOAgent(name="bot")
agent.train()


Timestep      64 | Episode   20 | Reward:     0.41
Timestep     128 | Episode   40 | Reward:     0.84
Timestep     193 | Episode   60 | Reward:     1.54
Timestep     257 | Episode   80 | Reward:     1.65
Timestep     320 | Episode  100 | Reward:     0.00
Timestep     383 | Episode  120 | Reward:     0.50
Timestep     450 | Episode  140 | Reward:     0.48
Timestep     516 | Episode  160 | Reward:     0.21
Timestep     582 | Episode  180 | Reward:     1.50
Timestep     647 | Episode  200 | Reward:     3.32
Timestep     711 | Episode  220 | Reward:     1.51
Timestep     774 | Episode  240 | Reward:     0.28
Timestep     839 | Episode  260 | Reward:     0.00
Timestep     905 | Episode  280 | Reward:     0.19
Timestep     974 | Episode  300 | Reward:     2.70
Timestep    1042 | Episode  320 | Reward:     2.19
Timestep    1106 | Episode  340 | Reward:     2.07
Timestep    1173 | Episode  360 | Reward:     2.77
Timestep    1244 | Episode  380 | Reward:     4.45
Timestep    1312 | Episode  400

In [21]:
results = agent.evaluate_model(n_episodes=100)
print(results)

{'episode_rewards': [39.53691757202148, 39.221110343933105, 39.21678623199463, 39.221576499938976, 39.080583839416505, 39.17689701080322, 39.37937297821045, 39.379180755615224, 39.202993392944336, 39.42909206390381, 39.27953990936279, 39.17631965637207, 39.09726387023926, 39.47314689636231, 39.51661384582519, 39.31651863098145, 39.13646884918212, 39.520813293457024, 39.319875335693354, 39.45408344268798, 39.34832778930664, 39.499137687683096, 39.1864059829712, 39.07388732910157, 39.28644252777099, 39.49820411682128, 39.28272563934326, 39.43786567687988, 39.05613960266113, 39.37731986999511, 39.25705909729004, 39.26525615692139, 39.52185119628906, 39.549802207946776, 39.378680229187005, 39.40918094635009, 39.45294456481934, 39.42279235839843, 39.23998142242431, 39.429130058288564, 39.41846343994141, 39.16487590789795, 39.31972137451172, 39.3374765777588, 39.30315341949463, 39.38502162933351, 39.405396423339845, 39.13702590942383, 39.31558704376221, 39.53320713043213, 39.47848903656007, 

In [22]:
agent.play()

Demand: 10, Price: 42.31, Reward: 2.24, Sold: 10, Revenue: 423.12
Demand: 6, Price: 42.74, Reward: 1.89, Sold: 6, Revenue: 256.44
Demand: 6, Price: 42.93, Reward: 3.03, Sold: 6, Revenue: 257.58
Demand: 2, Price: 43.19, Reward: 3.05, Sold: 2, Revenue: 86.37
Demand: 4, Price: 43.39, Reward: 0.77, Sold: 4, Revenue: 173.55
Demand: 3, Price: 43.39, Reward: 2.69, Sold: 3, Revenue: 130.16
Demand: 4, Price: 43.68, Reward: 1.16, Sold: 4, Revenue: 174.72
Demand: 5, Price: 43.79, Reward: 1.55, Sold: 5, Revenue: 218.95
Demand: 4, Price: 43.99, Reward: 1.56, Sold: 4, Revenue: 175.96
Demand: 4, Price: 44.19, Reward: 2.74, Sold: 4, Revenue: 176.77
Demand: 5, Price: 44.55, Reward: 1.98, Sold: 5, Revenue: 222.77
Demand: 4, Price: 44.78, Reward: 2.39, Sold: 4, Revenue: 179.13
Demand: 3, Price: 45.11, Reward: 2.81, Sold: 3, Revenue: 135.33
Demand: 4, Price: 45.62, Reward: 0.81, Sold: 4, Revenue: 182.50
Demand: 4, Price: 45.80, Reward: 2.45, Sold: 4, Revenue: 183.21
Demand: 2, Price: 46.46, Reward: 2.07, 

____
### 5. Training the TD3 Model
- TD3 uses 3 networks
    1. Actor Network
        - Outputs a single exact action
        - `state → action`
    2. Critic Network (2 critic networks)
        - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as PPO)
    3. Replay Buffer
        - To store experiences and reuse them, making the TD3 highly sample efficient
        - Allow TD3 to learn from past experiences repeatedly
    4. Target Networks
        - the original network that does not change, acts as a **control** to compare the adjusted networks to 
##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`

Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)

#### 5.1 Creating Actor and Critic Networks

In [ ]:
class TD3Actor(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh()
        )
        
    def forward(self, obs): #needs to be overridden
        return self.net(obs)

In [25]:
class TD3Critic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(obs_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, obs, action):
        x = torch.cat([obs, action], dim=-1)
        return self.net(x)

##### 5.11 Explanation of code

```python
self.net = nn.Sequential( #actor network
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Tanh())
```

- `nn.sequential()` ensures that the following layers run in sequence
- choice of functions used:
- `nn.Linear(obs_dim, 64)` -> y = Wx + b ; used to expand the 3 value state into 64 values
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.ReLU()` -> y = ReLu(x) = max(0, x)
    - The ReLU function reads every value and limits it to only non-negative number
- `nn.Linear(64, action_dim)` -> y = Wx + b
    - converts the 64 dimensional vector back into a 1 dimensional value, the exact price
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer
- `forward(self, obs)` is the method used to obtain the action based off the observation 

```python
self.net = nn.Sequential( #critic network
            nn.Linear(obs_dim + action_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1))
```
- `nn.Linear(obs_dim + action_dim, 64)`
    - the neural network needs to read both the observation and the action in order to come up with a measurement for future rewards (critic value)
    - expands it out into 64 values
- `nn.Linear(64, 1)`
    - the 64 values are then compressed back to produce 1 value, the Q-value which is the measurement of future rewards
- `forward(self, obs, action)` is the method used to obtain the Q-value, the estimate of discounted future rewards
    - `x = torch.cat([obs, action], dim = -1)` concatenates the observation and the action into 1 tensor, which is then used as the input for the neural network when `self.net(x)` is called

##### 5.12 Overarching view: PPO vs TD3
- Choice of hidden layers:
    - PPO's action network outputs a distribution instead and then samples it to obtain an action 
    - Therefore, the use of `tanh()` in the hidden layers provides a bound for the distribution
    - However, since the TD3' actor network outputs an action directly, there is no need to limit the bounds with a `tanh()` function in the hidden layers
    - Instead, `ReLU()` can be used for the unboundedness to improve training efficiency
    - If `tanh()` were to be used, when input values are very large or very small, the gradient approaches zero, making the weights in early layers small and barely updated during backprop
- Difference in Critic
    - PPO's critic only estimated and answered "how good is this state", therefore it only read the state
    - TD3's critic estimates and answers "how good is this action in this state", therefore it reads both the state and action 
    - Additionally, TD3 has two critic networks to take the minimum Q-value to prevent overestimation bias

#### 5.2 Creating ReplayBuffer
- used to store up to 100,000 transitions and do not clear out, overriding the oldest entry when full
- since TD3 answers "how good is this action in this state", it does not require episode tracking 

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=100000): 
        self.capacity = capacity
        self.buffer = []
        self.position = 0 #tracks where to write the next transition

    def add(self, obs, action, reward, next_obs, done):
        transition = (obs, action, reward, next_obs,done)
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        batch = np.random.sample(self.buffer, batch_size)
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (torch.tensor(np.array(obs), dtype=torch.float32),
                torch.tensor(np.array(actions), dtype=torch.float32),
                torch.tensor(np.array(rewards), dtype=torch.float32).unsqueeze(1),
                torch.tensor(np.array(next_obs), dtype=torch.float32),
                torch.tensor(np.array(dones), dtype=torch.float32).unsqueeze(1))
    def __len__(self):
        return len(self.buffer)

##### 5.21 Explanation of code

```python
def add(self, obs, action, reward, next_obs, done):
    transition = (obs, action, reward, next_obs,done)
    if len(self.buffer) < self.capacity:
        self.buffer.append(transition)
    else:
        self.buffer[self.position] = transition
        self.position = (self.position + 1) % self.capacity
```
- `if len(self.buffer) < self.capacity` : if the buffer still has space
    - `self.buffer.append(transition)` : add the values in and just grow the buffer
- if it is full, rewrites the first output:
    - `self.buffer[self.position] = transition` : overrides the first value as `self.position = 0` until updated
    - `self.position = (self.position + 1) % self.capacity` : advances the write-head by 1

```python
def sample(self, batch_size):
    batch = np.random.sample(self.buffer, batch_size)
    obs, actions, rewards, next_obs, dones = zip(*batch)
    return (torch.tensor(np.array(obs), dtype=torch.float32),
            torch.tensor(np.array(actions), dtype=torch.float32),
            torch.tensor(np.array(rewards), dtype=torch.float32).unsqueeze(1),
            torch.tensor(np.array(next_obs), dtype=torch.float32),
            torch.tensor(np.array(dones), dtype=torch.float32).unsqueeze(1))
```
- `batch = np.random.sample(self.buffer, batch_size)` : draw a batch_size number of transitions uniformly at random
- `obs, actions, rewards, next_obs, dones = zip(*batch)` : unpacks the list of tuples into 5 seperate tuples, one per field


In [ ]:
class TD3Agent:
    def __init__(self, env=None, gamma=0.99, tau=0.005, policy_noise=0.2, noise_clip=0.5,   # max magnitude of that noise
                 policy_delay=2, expl_noise=0.1, batch_size=256, buffer_capacity=100_000,
                 lr=1e-3):

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.env = env if env is not None else DynamicPricingEnv()
        
        # price range for rescaling tanh output (-1,1) → (5, 50)
        self.action_low  = float(self.env.action_space.low[0])   # 5.0
        self.action_high = float(self.env.action_space.high[0])  # 50.0

        # main networks
        self.actor   = TD3Actor(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)
        self.critic1 = TD3Critic(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)
        self.critic2 = TD3Critic(self.env.observation_space.shape[0],  self.env.action_space.shape[0]).to(self.device)

        # target networks — frozen copies, updated slowly via polyak
        self.tgt_actor   = copy.deepcopy(self.actor)
        self.tgt_critic1 = copy.deepcopy(self.critic1)
        self.tgt_critic2 = copy.deepcopy(self.critic2)

        # target networks never receive gradient updates directly
        for net in [self.tgt_actor, self.tgt_critic1, self.tgt_critic2]:
            for param in net.parameters():
                param.requires_grad = False

        # optimizers
        self.actor_opt   = optim.Adam(self.actor.parameters(),   lr=lr)
        self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=lr)

        self.replay_buffer = ReplayBuffer(capacity=buffer_capacity)

        # hyperparameters
        self.gamma        = gamma
        self.tau          = tau
        self.policy_noise = policy_noise
        self.noise_clip   = noise_clip
        self.policy_delay = policy_delay
        self.expl_noise   = expl_noise
        self.batch_size   = batch_size

        self.total_updates = 0  # tracks how many critic updates done, for policy_delay

    # ------------------------------------------------------------------ #
    #  HELPER: rescale tanh output (-1, 1) → (action_low, action_high)   #
    # ------------------------------------------------------------------ #
    def _rescale(self, raw_action):
        # maps (-1,1) linearly to (5, 50)
        return self.action_low + (raw_action + 1.0) * 0.5 * (self.action_high - self.action_low)

    # ------------------------------------------------------------------ #
    #  ACT: given an observation, return a price                          #
    # ------------------------------------------------------------------ #
    def act(self, obs, add_noise=True):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=self.device).unsqueeze(0)

        self.actor.eval()
        with torch.no_grad():
            raw = self.actor(obs_t)         # tanh output in (-1, 1)
        self.actor.train()

        price = self._rescale(raw.cpu().numpy()[0])   # shape (1,), value in (5, 50)

        if add_noise:
            # gaussian exploration noise, scaled to action range
            noise = np.random.normal(0, self.expl_noise *
                                     (self.action_high - self.action_low),
                                     size=price.shape)
            price = np.clip(price + noise, self.action_low, self.action_high)

        return price.astype(np.float32)

    # ------------------------------------------------------------------ #
    #  UPDATE: one gradient step on a sampled minibatch                   #
    # ------------------------------------------------------------------ #
    def _update(self):
        # unpack a random minibatch from the replay buffer
        obs, actions, rewards, next_obs, dones = self.replay_buffer.sample(self.batch_size)

        # move everything to the correct device
        obs      = obs.to(self.device)
        actions  = actions.to(self.device)
        rewards  = rewards.to(self.device)
        next_obs = next_obs.to(self.device)
        dones    = dones.to(self.device)

        # ---- CRITIC UPDATE ------------------------------------------ #
        with torch.no_grad():
            # target actor suggests the next action from next_obs
            raw_next = self.tgt_actor(next_obs)              # (-1, 1)

            # rescale to price range as a tensor for critic input
            next_actions = self.action_low + (raw_next + 1.0) * 0.5 * \
                           (self.action_high - self.action_low)

            # target policy smoothing: add clipped noise to next action
            # this prevents the actor from exploiting narrow Q-value peaks
            noise = torch.randn_like(next_actions) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_actions = (next_actions + noise).clamp(self.action_low, self.action_high)

            # twin critics evaluate (next_obs, next_actions)
            q1_next = self.tgt_critic1(next_obs, next_actions)
            q2_next = self.tgt_critic2(next_obs, next_actions)

            # take the minimum to avoid overestimation (the "Twin" trick)
            q_next = torch.min(q1_next, q2_next)

            # bellman target: if episode ended (done=1), no future reward
            target_q = rewards + self.gamma * (1.0 - dones) * q_next

        # compute current Q estimates and MSE against the target
        q1_current = self.critic1(obs, actions)
        q2_current = self.critic2(obs, actions)

        critic1_loss = nn.functional.mse_loss(q1_current, target_q)
        critic2_loss = nn.functional.mse_loss(q2_current, target_q)

        # backpropagate both critics independently
        self.critic1_opt.zero_grad()
        critic1_loss.backward()
        self.critic1_opt.step()

        self.critic2_opt.zero_grad()
        critic2_loss.backward()
        self.critic2_opt.step()

        self.total_updates += 1

        # ---- DELAYED ACTOR UPDATE ------------------------------------ #
        # only update actor every policy_delay critic updates
        if self.total_updates % self.policy_delay == 0:

            # actor loss: maximise Q1(obs, actor(obs))
            # gradient ascent on Q → gradient descent on negative Q
            raw_actions   = self.actor(obs)
            actor_actions = self.action_low + (raw_actions + 1.0) * 0.5 * \
                            (self.action_high - self.action_low)

            actor_loss = -self.critic1(obs, actor_actions).mean()

            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()

            # ---- POLYAK UPDATE on target networks -------------------- #
            # slowly blend main network weights into target networks
            for main, target in [(self.actor,   self.tgt_actor),
                                  (self.critic1, self.tgt_critic1),
                                  (self.critic2, self.tgt_critic2)]:
                for p_main, p_tgt in zip(main.parameters(), target.parameters()):
                    p_tgt.data.mul_(1.0 - self.tau)
                    p_tgt.data.add_(self.tau * p_main.data)

    # ------------------------------------------------------------------ #
    #  TRAIN: main training loop                                          #
    # ------------------------------------------------------------------ #
    def train(self, total_timesteps=200_000, learning_starts=1_000, log_every=20):
        print(f"Training TD3 on: {self.device}")
        obs, _ = self.env.reset()
        episode_reward = 0.0
        episode_count  = 0

        for timestep in range(1, total_timesteps + 1):

            # before learning_starts, take random actions to pre-fill buffer
            if timestep < learning_starts:
                action = self.env.action_space.sample()
            else:
                action = self.act(obs, add_noise=True)

            next_obs, reward, terminated, truncated, _ = self.env.step(action)
            done = terminated or truncated

            # store transition — use terminated (not done) for the done flag
            # so that a truncated episode doesn't incorrectly zero out future rewards
            self.replay_buffer.add(obs, action, reward, next_obs, float(terminated))

            episode_reward += reward
            obs = next_obs

            if done:
                episode_count += 1
                if episode_count % log_every == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0.0
                obs, _ = self.env.reset()

            # only start learning once the buffer has enough samples
            if timestep >= learning_starts:
                self._update()

        print("Training complete.")
        return self

    # ------------------------------------------------------------------ #
    #  EVALUATE                                                           #
    # ------------------------------------------------------------------ #
    def evaluate(self, n_episodes=100):
        rewards, inventories, steps = [], [], []

        for _ in range(n_episodes):
            env = DynamicPricingEnv()
            obs, _ = env.reset()
            done = False
            total_reward = 0.0

            while not done:
                action = self.act(obs, add_noise=False)   # deterministic at eval
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                done = terminated or truncated

            rewards.append(total_reward)
            inventories.append(env.inventory)
            steps.append(env.step_count)

        print(f"Episodes : {n_episodes}")
        print(f"Mean reward : {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")
        print(f"Reward range: [{np.min(rewards):.2f}, {np.max(rewards):.2f}]")
        print(f"Mean ending inventory: {np.mean(inventories):.2f}")
        print(f"Mean steps : {np.mean(steps):.2f}")